## Modeling - Decision Tree Regressor

Target variable: **ClaimNb**


### From-Scratch Implementation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split

# Preprocessing
df=pd.read_csv('Project_description_and_data/claims_train.csv')

df['VehAge_log'] = np.log(df['VehAge'] + 1)
df['DrivAge_log'] = np.log(df['DrivAge'])
df['Density_log'] = np.log(df['Density'])

categorical_cols = ['VehBrand', 'VehGas', 'Area', 'Region']
for col in categorical_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

df['Exposure_adj'] = df['Exposure'].clip(upper=1.0)

features = ['Exposure_adj','VehBrand','VehGas','VehPower',
            'VehAge_log','DrivAge_log','Area','Density_log',
            'Region','BonusMalus']
X = df[features].values
y = df['ClaimNb'].values

y_binned = pd.cut(y, bins=[-1,0,1,100], labels=[0,1,2])     # Group into 3 bins: 0 claims, 1 claim, ≥2 claims

# Stratified split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, val_idx in sss.split(X, y_binned):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

# # Manual train/validation split
# np.random.seed(42)
# indices = np.arange(len(X))
# np.random.shuffle(indices)
# split = int(0.8 * len(X))
# train_idx, val_idx = indices[:split], indices[split:]
# X_train, X_val = X[train_idx], X[val_idx]
# y_train, y_val = y[train_idx], y[val_idx]


# Not from-scratch split:
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


# Decision Tree Regressor from scratch
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
    
    def is_leaf(self):
        return self.value is not None

def mse(y):
    if len(y) == 0:
        return 0
    mean = np.mean(y)
    return np.mean((y - mean) ** 2)

def weighted_mse(y_left, y_right):
    n = len(y_left) + len(y_right)
    return (len(y_left)/n) * mse(y_left) + (len(y_right)/n) * mse(y_right)

class DecisionTreeRegressorScratch:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
    
    def fit(self, X, y):
        self.root = self._build_tree(X, y, depth=0)
        return self
    
    def _build_tree(self, X, y, depth):
        n_samples, n_features = X.shape

        # stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or n_samples < self.min_samples_split:
            leaf_value = np.mean(y)
            return Node(value=leaf_value)
        
        # find best split
        best_feature, best_threshold = self._best_split(X, y)
        if best_feature is None:
            leaf_value = np.mean(y)
            return Node(value=leaf_value)
        
        # split data
        left_indices = X[:, best_feature] <= best_threshold
        right_indices = X[:, best_feature] > best_threshold
        
        left_child = self._build_tree(X[left_indices], y[left_indices], depth+1)
        right_child = self._build_tree(X[right_indices], y[right_indices], depth+1)
        
        return Node(feature=best_feature, threshold=best_threshold,
                    left=left_child, right=right_child)
    
    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        if n_samples <= 1:
            return None, None
        
        best_mse = float('inf')
        best_feature, best_threshold = None, None
        
        for feature_idx in range(n_features):
            X_col = X[:, feature_idx]
            thresholds = np.unique(X_col)
            
            for i in range(len(thresholds) - 1):
                threshold = (thresholds[i] + thresholds[i+1]) / 2
                left_indices = X_col <= threshold
                right_indices = X_col > threshold
                
                left_y, right_y = y[left_indices], y[right_indices]
                w_mse = weighted_mse(left_y, right_y)
                
                if w_mse < best_mse:
                    best_mse = w_mse
                    best_feature = feature_idx
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
    
    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)


def mean_squared_error_manual(y_true, y_pred):
    errors = (y_true - y_pred) ** 2
    return np.mean(errors)

def mean_absolute_error_manual(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res/ss_tot

# Train and evaluate
scratch_tree = DecisionTreeRegressorScratch(max_depth=7, min_samples_split=200)
scratch_tree.fit(X_train, y_train)
y_val_pred_scratch = scratch_tree.predict(X_val)

print("From-scratch Decision Tree:")
print("Validation MSE:", mean_squared_error_manual(y_val, y_val_pred_scratch))
print("Validation MAE:", mean_absolute_error_manual(y_val, y_val_pred_scratch))
print("Validation R²:", r2_score_manual(y_val, y_val_pred_scratch))


From-scratch Decision Tree:
Validation MSE: 0.056095999711895835
Validation R²: 0.03215291851809965


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Preprocessing
df=pd.read_csv('Project_description_and_data/claims_train.csv')

df['Exposure_adj'] = df['Exposure'].clip(upper=1.0)

df['VehAge_1']    = df['VehAge'] + 1
df['VehAge_log']  = np.log(df['VehAge_1'])
df['DrivAge_log'] = np.log(df['DrivAge'])
df['Density_log'] = np.log(df['Density'])

pca_features = ['DrivAge_log', 'VehPower', 'VehAge_log', 'Density_log', 'BonusMalus']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[pca_features])

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

for i in range(3):
    df[f'PC{i+1}'] = X_pca[:, i]

cat_cols = ['VehBrand', 'Area', 'Region', 'VehGas']
for c in cat_cols:
    df[c] = df[c].astype('category')

df['DrivAgeLog_BonusMalus'] = df['DrivAge_log'] * df['BonusMalus']
df['VehPower_DensityLog'] = df['VehPower'] * df['Density_log']
df['VehAgeLog_Brand'] = df['VehAge_log'] * df['VehBrand'].cat.codes
df['BonusMalus_Exposure'] = df['BonusMalus'] * df['Exposure_adj']
df['VehPower_VehGas'] = df['VehPower'] * df['VehGas'].cat.codes

interaction_features = ['DrivAgeLog_BonusMalus', 'VehPower_DensityLog', 'VehAgeLog_Brand', 'BonusMalus_Exposure', 'VehPower_VehGas']

features = ['VehBrand', 'Region', 'PC1', 'PC2', 'PC3'] + interaction_features

for c in ['VehBrand', 'Region']:
    df[c] = df[c].cat.codes 

X = df[features].values
y = df['ClaimNb'].values

y_binned = pd.cut(y, bins=[-1,0,1,100], labels=[0,1,2])     # Group into 3 bins: 0 claims, 1 claim, ≥2 claims

# Stratified split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, val_idx in sss.split(X, y_binned):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

# Decision Tree Regressor from scratch
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
    
    def is_leaf(self):
        return self.value is not None

def mse(y):
    if len(y) == 0:
        return 0
    mean = np.mean(y)
    return np.mean((y - mean) ** 2)

def weighted_mse(y_left, y_right):
    n = len(y_left) + len(y_right)
    return (len(y_left)/n) * mse(y_left) + (len(y_right)/n) * mse(y_right)

class DecisionTreeRegressorScratch:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
    
    def fit(self, X, y):
        self.root = self._build_tree(X, y, depth=0)
        return self
    
    def _build_tree(self, X, y, depth):
        n_samples, n_features = X.shape

        # stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or n_samples < self.min_samples_split:
            leaf_value = np.mean(y)
            return Node(value=leaf_value)
        
        # find best split
        best_feature, best_threshold = self._best_split(X, y)
        if best_feature is None:
            leaf_value = np.mean(y)
            return Node(value=leaf_value)
        
        # split data
        left_indices = X[:, best_feature] <= best_threshold
        right_indices = X[:, best_feature] > best_threshold
        
        left_child = self._build_tree(X[left_indices], y[left_indices], depth+1)
        right_child = self._build_tree(X[right_indices], y[right_indices], depth+1)
        
        return Node(feature=best_feature, threshold=best_threshold,
                    left=left_child, right=right_child)
    
    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        if n_samples <= 1:
            return None, None
        
        best_mse = float('inf')
        best_feature, best_threshold = None, None
        
        for feature_idx in range(n_features):
            X_col = X[:, feature_idx]
            thresholds = np.unique(X_col)
            
            for i in range(len(thresholds) - 1):
                threshold = (thresholds[i] + thresholds[i+1]) / 2
                left_indices = X_col <= threshold
                right_indices = X_col > threshold
                
                left_y, right_y = y[left_indices], y[right_indices]
                w_mse = weighted_mse(left_y, right_y)
                
                if w_mse < best_mse:
                    best_mse = w_mse
                    best_feature = feature_idx
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
    
    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)


def mean_squared_error_manual(y_true, y_pred):
    errors = (y_true - y_pred) ** 2
    return np.mean(errors)

def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res/ss_tot


# Train and evaluate
scratch_tree = DecisionTreeRegressorScratch(max_depth=7, min_samples_split=200)
scratch_tree.fit(X_train, y_train)
y_val_pred_scratch = scratch_tree.predict(X_val)

print("From-scratch Decision Tree:")
print("Validation MSE:", mean_squared_error_manual(y_val, y_val_pred_scratch))
print("Validation R²:", r2_score_manual(y_val, y_val_pred_scratch))
print("Validation MAE:", mean_absolute_error(y_val, y_val_pred_scratch))

KeyboardInterrupt: 

### Reference Implementation

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Preprocessing with libraries
categorical_cols = ['VehBrand', 'VehGas', 'Area', 'Region']
for col in categorical_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

y_binned = pd.cut(y, bins=[-1,0,1,100], labels=[0,1,2])
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, val_idx in sss.split(X, y_binned):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]


# Hyperparameter tuning
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [10, 50, 200]
}

grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=3,                           # Cross‑validation folds
                                    # The training set is split into 3 equal parts; each part is used once as validation while the other two are used for training.
    scoring='r2',
    n_jobs=-1                       # Run the grid search in parallel across all available CPU cores (maximizes speed)
)
grid_search.fit(X_train, y_train)

print("Best hyperparameters:", grid_search.best_params_)

# Train final model
best_tree = grid_search.best_estimator_
best_tree.fit(X_train, y_train)

# Predict and evaluate
y_val_pred = best_tree.predict(X_val)

print("Reference Decision Tree Regressor (scikit-learn):")
print("Validation MSE:", mean_squared_error(y_val, y_val_pred))
print("Validation MAE:", mean_absolute_error(y_val, y_val_pred))
print("Validation R²:", r2_score(y_val, y_val_pred))


Best hyperparameters: {'max_depth': 7, 'min_samples_split': 200}
Reference Decision Tree Regressor (scikit-learn):
Validation MSE: 0.056095999711895835
Validation R²: 0.03215291851809965


##### Explaining why cv = 3 for grid search:

In [ ]:
# Grid search with cv=3
grid_search_cv3 = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=3,                           
    scoring='r2',
    n_jobs=-1
)
grid_search_cv3.fit(X_train, y_train)

print("Best params with cv=3:", grid_search_cv3.best_params_)
print("Best R² (cv=3):", grid_search_cv3.best_score_)

# Grid search with cv=5
grid_search_cv5 = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid_search_cv5.fit(X_train, y_train)

print("Best params with cv=5:", grid_search_cv5.best_params_)
print("Best R² (cv=5):", grid_search_cv5.best_score_)


# Evaluate best model from cv=5 on validation set
best_tree = grid_search_cv5.best_estimator_
y_val_pred = best_tree.predict(X_val)

print("Validation MSE:", mean_squared_error(y_val, y_val_pred))
print("Validation MAE:", mean_absolute_error(y_val, y_val_pred))
print("Validation R²:", r2_score(y_val, y_val_pred))

Best params with cv=3: {'max_depth': 7, 'min_samples_split': 200}
Best R² (cv=3): 0.02719436036883216
Best params with cv=5: {'max_depth': 7, 'min_samples_split': 200}
Best R² (cv=5): 0.027492180667830324
Validation MSE: 0.056095999711895835
Validation R²: 0.03215291851809965


In [ ]:
# Hyperparameter tuning loop
depth_values = [3, 5, 7, 10]
min_samples_values = [10, 50, 200]

results = []

print("Scratch Tree Results:")
for depth in depth_values:
    for min_samples in min_samples_values:
        scratch_tree = DecisionTreeRegressorScratch(max_depth=depth, min_samples_split=min_samples)
        scratch_tree.fit(X_train, y_train)
        y_val_pred = scratch_tree.predict(X_val)
        
        mse_val = mean_squared_error(y_val, y_val_pred)
        r2_val = r2_score(y_val, y_val_pred)
        
        results.append(("scratch", depth, min_samples, mse_val, r2_val))
        print(f"depth={depth}, min_samples={min_samples} -> MSE={mse_val:.4f}, R²={r2_val:.4f}")

print("\nscikit-learn Tree Results:")
for depth in depth_values:
    for min_samples in min_samples_values:
        ref_tree = DecisionTreeRegressor(max_depth=depth, min_samples_split=min_samples, random_state=42)
        ref_tree.fit(X_train, y_train)
        y_val_pred_ref = ref_tree.predict(X_val)
        
        mse_val_ref = mean_squared_error(y_val, y_val_pred_ref)
        r2_val_ref = r2_score(y_val, y_val_pred_ref)
        
        results.append(("sklearn", depth, min_samples, mse_val_ref, r2_val_ref))
        print(f"depth={depth}, min_samples={min_samples} -> MSE={mse_val_ref:.4f}, R²={r2_val_ref:.4f}")


Scratch Tree Results:
depth=3, min_samples=10 -> MSE=0.0569, R²=0.0178
depth=3, min_samples=50 -> MSE=0.0569, R²=0.0178
depth=3, min_samples=200 -> MSE=0.0569, R²=0.0178
depth=5, min_samples=10 -> MSE=0.0565, R²=0.0254
depth=5, min_samples=50 -> MSE=0.0565, R²=0.0254
depth=5, min_samples=200 -> MSE=0.0565, R²=0.0255
depth=7, min_samples=10 -> MSE=0.0561, R²=0.0315
depth=7, min_samples=50 -> MSE=0.0561, R²=0.0320
depth=7, min_samples=200 -> MSE=0.0561, R²=0.0322
depth=10, min_samples=10 -> MSE=0.0568, R²=0.0205
depth=10, min_samples=50 -> MSE=0.0565, R²=0.0253
depth=10, min_samples=200 -> MSE=0.0562, R²=0.0308

scikit-learn Tree Results:
depth=3, min_samples=10 -> MSE=0.0569, R²=0.0178
depth=3, min_samples=50 -> MSE=0.0569, R²=0.0178
depth=3, min_samples=200 -> MSE=0.0569, R²=0.0178
depth=5, min_samples=10 -> MSE=0.0565, R²=0.0254
depth=5, min_samples=50 -> MSE=0.0565, R²=0.0254
depth=5, min_samples=200 -> MSE=0.0565, R²=0.0255
depth=7, min_samples=10 -> MSE=0.0561, R²=0.0315
depth=7, m